In [1]:
from pathlib import Path

import polars as pl
import yaml
from shapely import Polygon
from common import daterange
import geopandas as gpd

with open("../config.copenhagen_sample.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

In [2]:
DIR = f"{config['dirs']['output']}/{config['city']}/partitioned"

In [3]:
lon_min = 180
lon_max = -180
lat_min = 90
lat_max = -90
for i in range(256):
    data = pl.read_parquet(f"{DIR}/{i:02x}/pings.parquet")
    omin = data["longitude"].min()
    if omin < lon_min:
        lon_min = omin
    omax = data["longitude"].max()
    if omax > lon_max:
        lon_max = omax
    amin = data["latitude"].min()
    if amin < lat_min:
        lat_min = amin
    amax = data["latitude"].max()
    if amax > lat_max:
        lat_max = amax

    del data
# runs 6 min

In [4]:
bbox = Polygon(
    [(lon_min, lat_min), (lon_max, lat_min), (lon_max, lat_max), (lon_min, lat_max)]
)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs=4326)
Path(f"{config['dirs']['output']}/{config['city']}").mkdir(parents=True, exist_ok=True)
bbox_gdf.to_file(f"{config['dirs']['output']}/{config['city']}/bounding_box.geojson")